# Pipeline de Engenharia de Dados: Extração de Features & Regras de Negócio com IA Local (Ollama)

Este notebook realiza engenharia reversa sobre o código-fonte do **OWASP Juice Shop** para gerar um dataset estruturado de **User Stories e Regras de Negócio** (no formato `nome da feature` + `Action`), respeitando o limite máximo de **50 MB**.

### Fluxo do Pipeline:
1. **Configuração Interativa:** Seleção gráfica dos diretórios de entrada/saída e escolha do modelo local no **Ollama**.
2. **Análise de Código-Fonte com IA:** Varredura dos módulos de back-end (`routes/`) e front-end (`frontend/`) para extrair regras comportamentais, validações e jornadas de usuário.
3. **Expansão e Geração do Dataset (< 50 MB):** Geração controlada em lotes até atingir o teto de tamanho configurado, salvando os dados prontos para uso em pipelines de ETL ou NLP.


In [13]:
import os
import requests
import tkinter as tk
from tkinter import filedialog
import ipywidgets as widgets
from IPython.display import display

# ==========================================
# 1. CAMINHOS PADRÃO E CAIXAS DE DIÁLOGO
# ==========================================
DEFAULT_REPO_PATH = r"D:\projetos\juice-shop"
DEFAULT_DEST_PATH = r"C:\Users\alexs\Desktop\MBA -  Inteligência Artificial Aplicada\4 bimestre\Processamento de Linguagem Natural (PLN)\trabalho"

def selecionar_pasta(titulo: str, caminho_inicial: str) -> str:
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    
    # Abre a janela já dentro do diretório padrão (se ele existir)
    dir_inicial = caminho_inicial if os.path.exists(caminho_inicial) else os.getcwd()
    caminho = filedialog.askdirectory(title=titulo, initialdir=dir_inicial)
    root.destroy()
    
    # Se o usuário cancelar a janela, usa o caminho padrão definido
    return caminho if caminho else caminho_inicial

print("1️⃣ Aguardando confirmação/seleção da pasta do repositório (Entrada)...")
REPO_PATH = selecionar_pasta(
    "1. Confirme ou selecione a pasta do repositório OWASP Juice Shop",
    DEFAULT_REPO_PATH
)

print("2️⃣ Aguardando confirmação/seleção da pasta de destino do dataset (Saída)...")
pasta_destino_base = selecionar_pasta(
    "2. Confirme ou selecione onde deseja salvar o dataset gerado",
    DEFAULT_DEST_PATH
)

# Cria a subpasta organizada dentro do destino escolhido
OUTPUT_DIR = os.path.join(pasta_destino_base, "dataset_juice_shop")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_SIZE_MB = 48.0      # Limite de segurança abaixo de 50 MB
OUTPUT_FORMAT = "csv"   # Opções: "csv" ou "json"

print(f"\n✅ Repositório (Entrada): {REPO_PATH}")
print(f"📂 Destino do Dataset (Saída): {OUTPUT_DIR}\n")

# ==========================================
# 2. VERIFICAÇÃO DO OLLAMA E COMBO BOX DE MODELOS
# ==========================================
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_URL = f"{OLLAMA_BASE_URL}/api/generate"
OLLAMA_MODEL = None

def iniciar_seletor_ollama():
    global OLLAMA_MODEL
    try:
        resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        resp.raise_for_status()
        modelos = [m["name"] for m in resp.json().get("models", [])]
        
        if not modelos:
            raise RuntimeError("Servidor Ollama ativo, mas nenhum modelo encontrado. Rode 'ollama pull llama3' no terminal.")
            
        print("🟢 Servidor Ollama está ATIVO! Selecione o modelo na caixa abaixo:")
        
        # Define o primeiro modelo da lista como padrão inicial
        OLLAMA_MODEL = modelos[0]
        
        dropdown_modelos = widgets.Dropdown(
            options=modelos,
            value=modelos[0],
            description='Modelo IA:',
            disabled=False,
            layout=widgets.Layout(width='400px')
        )
        
        status_label = widgets.HTML(value=f"<b>Modelo ativo:</b> <code>{OLLAMA_MODEL}</code>")
        
        def ao_mudar_modelo(change):
            global OLLAMA_MODEL
            if change['type'] == 'change' and change['name'] == 'value':
                OLLAMA_MODEL = change['new']
                status_label.value = f"<b>Modelo ativo:</b> <code>{OLLAMA_MODEL}</code>"
                
        dropdown_modelos.observe(ao_mudar_modelo)
        display(widgets.VBox([dropdown_modelos, status_label]))

    except requests.exceptions.ConnectionError:
        print("🔴 ERRO: O servidor Ollama NÃO está acessível em http://localhost:11434.")
        print("👉 Dica: Abra o terminal e execute 'ollama serve' ou inicie o aplicativo do Ollama.")
        raise

iniciar_seletor_ollama()


1️⃣ Aguardando confirmação/seleção da pasta do repositório (Entrada)...
2️⃣ Aguardando confirmação/seleção da pasta de destino do dataset (Saída)...

✅ Repositório (Entrada): D:/projetos/juice-shop
📂 Destino do Dataset (Saída): C:/Users/alexs/Desktop/MBA -  Inteligência Artificial Aplicada/4 bimestre/Processamento de Linguagem Natural (PLN)/trabalho\dataset_juice_shop

🟢 Servidor Ollama está ATIVO! Selecione o modelo na caixa abaixo:


## Etapa 1: Análise de Código-Fonte e Extração de Features via Ollama

Nesta etapa, o pipeline lê os arquivos de back-end (`routes/`) e interface (`frontend/`) agrupados por domínio funcional e solicita ao **Ollama** que extraia as funcionalidades e regras de negócio no formato exato:
* `nome da feature`: Nome funcional do módulo ou tela.
* `Action`: User Story combinada com as regras de negócio, validações e comportamentos visuais extraídos do código.


In [14]:
import os
import glob
import json
import requests

def coletar_modulos_juice_shop(repo_path: str) -> dict:
    """Agrupa trechos de código por módulo funcional (Back-end e Front-end)."""
    if not os.path.exists(os.path.join(repo_path, "routes")):
        subpastas = glob.glob(os.path.join(repo_path, "*", "routes"))
        if subpastas:
            repo_path = os.path.dirname(subpastas)

    modulos = {
        "Carteira Digital e Pagamentos": glob.glob(os.path.join(repo_path, "routes", "*wallet*.ts")) + glob.glob(os.path.join(repo_path, "frontend", "src", "app", "*wallet*", "*.ts")),
        "Carrinho e Pedidos": glob.glob(os.path.join(repo_path, "routes", "*basket*.ts")) + glob.glob(os.path.join(repo_path, "routes", "*order*.ts")),
        "Assinatura Deluxe": glob.glob(os.path.join(repo_path, "routes", "*deluxe*.ts")) + glob.glob(os.path.join(repo_path, "frontend", "src", "app", "*deluxe*", "*.ts")),
        "Autenticação e Perfil": glob.glob(os.path.join(repo_path, "routes", "*login*.ts")) + glob.glob(os.path.join(repo_path, "routes", "*user*.ts")),
        "Cupons e Descontos": glob.glob(os.path.join(repo_path, "routes", "*coupon*.ts")),
        "Avaliações e Reclamações": glob.glob(os.path.join(repo_path, "routes", "*review*.ts")) + glob.glob(os.path.join(repo_path, "routes", "*complaint*.ts")),
        "Reciclagem de Embalagens": glob.glob(os.path.join(repo_path, "routes", "*recycle*.ts")),
    }

    contextos = {}
    for nome_modulo, arquivos in modulos.items():
        texto_modulo = []
        for arq in arquivos[:3]:
            try:
                with open(arq, "r", encoding="utf-8") as f:
                    texto_modulo.append(f"// Arquivo: {os.path.basename(arq)}\n" + f.read()[:2000])
            except Exception:
                pass
        if texto_modulo:
            contextos[nome_modulo] = "\n".join(texto_modulo)
    return contextos

def gerar_seeds_com_ollama(contextos: dict) -> list:
    """Usa o Ollama para extrair pares de 'nome da feature' e 'Action' baseados no código."""
    modelo_atual = OLLAMA_MODEL if isinstance(OLLAMA_MODEL, list) else OLLAMA_MODEL
    seeds_extraidas = []

    for modulo, codigo in contextos.items():
        prompt = f"""
Você é um Analista de Requisitos e Engenheiro de Dados. Analise o código abaixo do módulo "{modulo}" do sistema OWASP Juice Shop.

Extraia 4 funcionalidades (features) reais presentes nesse código e escreva cada uma EXATAMENTE seguindo o padrão do exemplo abaixo:

Exemplo esperado:
nome da feature: Painel Financeiro
Action: "Como usuário logado, quero acessar meu painel financeiro para ver meu saldo. O sistema deve ocultar o saldo por padrão. Se o saldo for negativo, o texto deve ficar vermelho."

Regras obrigatórias:
- Retorne APENAS um array JSON válido, sem texto antes ou depois.
- Cada objeto do JSON deve ter as chaves: "nome_da_feature" e "action".
- O campo "action" deve combinar a User Story ("Como [perfil], quero [ação] para [objetivo].") seguida das regras de negócio e comportamento de tela/validação encontradas no código.

Código do módulo:
{codigo[:4500]}
"""
        payload = {
            "model": modelo_atual,
            "prompt": prompt,
            "format": "json",
            "stream": False
        }

        try:
            print(f"🤖 Analisando módulo '{modulo}' com {modelo_atual}...")
            resp = requests.post(OLLAMA_URL, json=payload, timeout=180)
            resp.raise_for_status()
            resposta_texto = resp.json().get("response", "{}")
            dados = json.loads(resposta_texto)

            itens = dados if isinstance(dados, list) else next((v for v in dados.values() if isinstance(v, list)), [])
            for item in itens:
                if "nome_da_feature" in item and "action" in item:
                    seeds_extraidas.append({
                        "modulo": modulo,
                        "nome_da_feature": item["nome_da_feature"].strip(),
                        "action": item["action"].strip()
                    })
        except Exception as e:
            print(f"⚠️ Aviso ao processar '{modulo}': {e}")

    return seeds_extraidas

# Executa a extração inteligente com o Ollama
contextos_codigo = coletar_modulos_juice_shop(REPO_PATH)
seeds_features = gerar_seeds_com_ollama(contextos_codigo)

# Fallback de segurança caso algum modelo local não retorne JSON válido
if not seeds_features:
    seeds_features = [
        {
            "modulo": "Carteira Digital e Pagamentos",
            "nome_da_feature": "Painel Financeiro",
            "action": "Como usuário logado, quero acessar meu painel financeiro para ver meu saldo. O sistema deve ocultar o saldo por padrão. Se o saldo for negativo, o texto deve ficar vermelho."
        },
        {
            "modulo": "Carrinho e Pedidos",
            "nome_da_feature": "Aplicação de Cupom Promocional",
            "action": "Como cliente no checkout, quero inserir um código de cupom para obter desconto no pedido. O sistema deve validar a data de expiração do cupom. Se o cupom for inválido ou expirado, exibir mensagem de erro em vermelho."
        },
        {
            "modulo": "Assinatura Deluxe",
            "nome_da_feature": "Adesão ao Plano Deluxe",
            "action": "Como usuário autenticado, quero assinar o plano Deluxe para obter preços reduzidos e frete grátis. O sistema deve verificar se o saldo da carteira é suficiente. Se o pagamento for aprovado, atualizar imediatamente o perfil do usuário para Deluxe."
        }
    ]

print(f"\n✅ Total de regras base extraídas do código: {len(seeds_features)}")
for s in seeds_features[:3]:
    print(f"\nnome da feature: {s['nome_da_feature']}\nAction: \"{s['action']}\"")


🤖 Analisando módulo 'Carteira Digital e Pagamentos' com gemma4:12b...
⚠️ Aviso ao processar 'Carteira Digital e Pagamentos': Unterminated string starting at: line 9 column 17 (char 460)
🤖 Analisando módulo 'Carrinho e Pedidos' com gemma4:12b...
⚠️ Aviso ao processar 'Carrinho e Pedidos': Expecting value: line 1 column 1 (char 0)
🤖 Analisando módulo 'Assinatura Deluxe' com gemma4:12b...
🤖 Analisando módulo 'Autenticação e Perfil' com gemma4:12b...
🤖 Analisando módulo 'Cupons e Descontos' com gemma4:12b...
🤖 Analisando módulo 'Avaliações e Reclamações' com gemma4:12b...
⚠️ Aviso ao processar 'Avaliações e Reclamações': Expecting value: line 1 column 1 (char 0)
🤖 Analisando módulo 'Reciclagem de Embalagens' com gemma4:12b...

✅ Total de regras base extraídas do código: 16

nome da feature: Upgrade via Carteira
Action: "Como cliente, quero realizar o upgrade para a Assinatura Deluxe utilizando minha carteira para processar o pagamento. O sistema deve verificar se a carteira existe e se o s

## Etapa 2: Geração e Expansão Controlada do Dataset (< 50 MB)

Nesta etapa final, utilizamos as regras extraídas pelo Ollama na Etapa 1 e geramos variações realistas de perfis, validações e comportamentos do sistema em lotes incrementais. O loop monitora o tamanho do arquivo em disco em tempo real e encerra a gravação automaticamente antes de ultrapassar **50 MB**.


In [16]:
import os
import random
import pandas as pd
from pathlib import Path

# Garante que a pasta de destino exista antes de iniciar a gravação
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def obter_tamanho_pasta_mb(pasta: str) -> float:
    total_bytes = sum(f.stat().st_size for f in Path(pasta).glob('*') if f.is_file())
    return total_bytes / (1024 * 1024)

perfis = [
    "usuário logado",
    "cliente Deluxe",
    "administrador da loja",
    "usuário visitante",
    "cliente com saldo ativo"
]

condicoes_extra = [
    "O sistema deve registrar um log de auditoria a cada tentativa.",
    "Se houver falha na validação, o botão de confirmação deve permanecer desabilitado.",
    "Os valores monetários devem ser exibidos com duas casas decimais.",
    "Em caso de tentativa não autorizada, o sistema deve retornar status HTTP 401.",
    "A interface deve atualizar os dados em tempo real sem recarregar a página."
]

arquivo_saida = os.path.join(OUTPUT_DIR, f"juice_shop_features_actions.{OUTPUT_FORMAT}")
if os.path.exists(arquivo_saida):
    os.remove(arquivo_saida)

batch_size = 5000
id_counter = 1
lote = 0

while True:
    tamanho_atual = obter_tamanho_pasta_mb(OUTPUT_DIR)
    if tamanho_atual >= MAX_SIZE_MB:
        print(f"\n✅ Dataset finalizado! Tamanho total: {tamanho_atual:.2f} MB (Limite: {MAX_SIZE_MB} MB)")
        break

    registros = []
    for _ in range(batch_size):
        base = random.choice(seeds_features)
        perfil_escolhido = random.choice(perfis)
        regra_complementar = random.choice(condicoes_extra)

        action_formatada = base["action"].replace("usuário logado", perfil_escolhido)
        if regra_complementar not in action_formatada:
            action_formatada = f"{action_formatada} {regra_complementar}"

        registros.append({
            "id": id_counter,
            "modulo": base["modulo"],
            "nome_da_feature": base["nome_da_feature"],
            "action": action_formatada,
            "registro_padrao": f'nome da feature: {base["nome_da_feature"]} Action:"{action_formatada}"'
        })
        id_counter += 1

    df_batch = pd.DataFrame(registros)
    modo = "w" if lote == 0 else "a"

    if OUTPUT_FORMAT == "csv":
        df_batch.to_csv(arquivo_saida, mode=modo, index=False, header=(lote == 0), encoding="utf-8")
    else:
        df_batch.to_json(arquivo_saida, orient="records", lines=True, mode=modo, force_ascii=False)

    lote += 1
    print(f"Lote {lote} | Registros: {id_counter - 1:,} | Tamanho: {obter_tamanho_pasta_mb(OUTPUT_DIR):.2f} MB", end="\r")

print(f"\n📂 Arquivo salvo em: {arquivo_saida}")
print(f"📊 Total de registros gerados: {id_counter - 1:,}")


Lote 13 | Registros: 65,000 | Tamanho: 48.49 MB
✅ Dataset finalizado! Tamanho total: 48.49 MB (Limite: 48.0 MB)

📂 Arquivo salvo em: C:/Users/alexs/Desktop/MBA -  Inteligência Artificial Aplicada/4 bimestre/Processamento de Linguagem Natural (PLN)/trabalho\dataset_juice_shop\juice_shop_features_actions.csv
📊 Total de registros gerados: 65,000
